# Formative 2 – Data Merge and Product Recommendation Model

**Goal:** merge the customer social-profile data with the customer transaction data, and train a model that predicts which product category a customer is likely to buy.

**Approach**
1. Clean each source on its own (IDs, duplicates, types, missing values).
2. Build features from **both** sources:
   - from the social profiles: engagement, purchase interest, platform, review sentiment;
   - from the transactions: each customer's *earlier* purchases (how many, average spend, average rating, which categories).
3. Merge them into one table with one row per transaction.
4. Compare models with grouped cross-validation, check the result against shuffled labels, and save the model and the cleaned data.

**Contents**
1. Setup
2. Load the data
3. Inspect the data
4. Clean each source
5. Build features and merge
6. Modelling
7. Recommendation engine
8. Save the cleaned data and the model

## 1. Setup

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 2. Load the data

The two datasets are read from CSV files in `data/raw/`. (To read them from Google Sheets in Colab instead, use the commented-out `gspread` code in the next cell.)

In [2]:
# from google.colab import auth
# import gspread
# from google.auth import default

# auth.authenticate_user()
# creds, _ = default()
# gc = gspread.authorize(creds)


# def load_sheet(name):
#     """Read the first worksheet of a Google Sheet into a DataFrame (first row = header)."""
#     values = gc.open(name).get_worksheet(0).get_all_values()
#     return pd.DataFrame(values[1:], columns=values[0])


df_sp = pd.read_csv("data/raw/customer_social_profiles.csv")
df_tr = pd.read_csv("data/raw/customer_transactions.csv")

## 3. Inspect the data

In [3]:
print(f"Social profiles: {df_sp.shape[0]} rows x {df_sp.shape[1]} columns")
df_sp.head()

Social profiles: 155 rows x 5 columns


,customer_id_new,social_media_platform,engagement_score,purchase_interest_score,review_sentiment
0,A178,LinkedIn,74,4.9,Positive
1,A190,Twitter,82,4.8,Neutral
2,A150,Facebook,96,1.6,Positive
3,A162,Twitter,89,2.6,Positive
4,A197,Twitter,92,2.3,Neutral


In [4]:
print(f"Transactions: {df_tr.shape[0]} rows x {df_tr.shape[1]} columns")
df_tr.head()

Transactions: 150 rows x 6 columns


,customer_id_legacy,transaction_id,purchase_amount,purchase_date,product_category,customer_rating
0,151,1001,408,2024-01-01,Sports,2.3
1,192,1002,332,2024-01-02,Electronics,4.2
2,114,1003,442,2024-01-03,Electronics,2.1
3,171,1004,256,2024-01-04,Clothing,2.8
4,160,1005,64,2024-01-05,Clothing,1.3


In [5]:
df_sp.info()
print()
df_tr.info()

<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id_new          155 non-null    str    
 1   social_media_platform    155 non-null    str    
 2   engagement_score         155 non-null    int64  
 3   purchase_interest_score  155 non-null    float64
 4   review_sentiment         155 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 6.2 KB

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id_legacy  150 non-null    int64  
 1   transaction_id      150 non-null    int64  
 2   purchase_amount     150 non-null    int64  
 3   purchase_date       150 non-null    str    
 4   product_category    150 non-null    str    
 5   customer_rating     140 non-n

In [6]:
print("Duplicate rows  - profiles:", df_sp.duplicated().sum(), "| transactions:", df_tr.duplicated().sum())
print("Missing values  - profiles:", df_sp.isna().sum().sum(), "| transactions:", df_tr.isna().sum().sum())
print("\nMissing per column (transactions):")
print(df_tr.isna().sum()[lambda s: s > 0])
print("\nValue ranges:")
pd.concat([df_sp.describe().T, df_tr[["purchase_amount", "customer_rating"]].describe().T])[["min", "max", "mean"]]

Duplicate rows  - profiles: 5 | transactions: 0
Missing values  - profiles: 0 | transactions: 10

Missing per column (transactions):
customer_rating    10
dtype: int64

Value ranges:


,min,max,mean
engagement_score,50.0,99.0,75.606452
purchase_interest_score,1.0,5.0,3.152903
purchase_amount,51.0,495.0,280.780000
customer_rating,1.0,5.0,2.985000


In [7]:
print("Platforms:   ", sorted(df_sp["social_media_platform"].unique()))
print("Sentiments:  ", sorted(df_sp["review_sentiment"].unique()))
print("Categories:  ", sorted(df_tr["product_category"].unique()))
print(f"\nProfile rows per customer:     {df_sp['customer_id_new'].value_counts().describe()[['min', 'max']].astype(int).to_dict()}")
print(f"Transactions per customer:    {df_tr['customer_id_legacy'].value_counts().describe()[['min', 'max']].astype(int).to_dict()}")

Platforms:    ['Facebook', 'Instagram', 'LinkedIn', 'TikTok', 'Twitter']
Sentiments:   ['Negative', 'Neutral', 'Positive']
Categories:   ['Books', 'Clothing', 'Electronics', 'Groceries', 'Sports']

Profile rows per customer:     {'min': 1, 'max': 7}
Transactions per customer:    {'min': 1, 'max': 6}


## 4. Clean each source

- **Customer IDs:** the profiles use `A178`-style IDs (`customer_id_new`), the transactions use plain numbers (`151`, `customer_id_legacy`). The legacy IDs get an `A` prefix so both tables share one `customer_id`.
- **Duplicates:** the profiles contain exact duplicate rows; these are dropped.
- **Dates:** `purchase_date` is converted to a real date.
- **Missing ratings:** left as missing (not filled with a fake `0`). They are handled inside the model pipeline instead.

In [8]:
profiles = (
    df_sp.drop_duplicates()
    .rename(columns={"customer_id_new": "customer_id"})
    .reset_index(drop=True)
)

transactions = df_tr.copy()
transactions["customer_id"] = "A" + transactions["customer_id_legacy"].astype(str).str.strip()
transactions["purchase_date"] = pd.to_datetime(transactions["purchase_date"])
transactions = (
    transactions.drop(columns="customer_id_legacy")
    .drop_duplicates()
    .sort_values(["purchase_date", "transaction_id"])
    .reset_index(drop=True)
)

CATEGORIES = sorted(transactions["product_category"].unique())

print(f"Profiles:     {len(df_sp)} -> {len(profiles)} rows after removing duplicates")
print(f"Transactions: {len(df_tr)} -> {len(transactions)} rows | unique transaction_id: {transactions['transaction_id'].is_unique}")
print(f"Customers with a profile: {profiles['customer_id'].nunique()} | with transactions: {transactions['customer_id'].nunique()}")

Profiles:     155 -> 150 rows after removing duplicates
Transactions: 150 -> 150 rows | unique transaction_id: True
Customers with a profile: 84 | with transactions: 75


## 5. Build features and merge

### 5.1 Profile features (one row per customer)

A customer can have several profile snapshots. Joining them all to every transaction would copy the same purchase into several rows with different feature values. Instead the snapshots are summarised into **one row per customer**: average engagement and interest, and the most common platform and sentiment.

In [9]:
profile_features = (
    profiles.groupby("customer_id")
    .agg(
        engagement_mean=("engagement_score", "mean"),
        interest_mean=("purchase_interest_score", "mean"),
        platform=("social_media_platform", lambda s: s.mode().iat[0]),
        sentiment=("review_sentiment", lambda s: s.mode().iat[0]),
    )
    .reset_index()
)
print(f"{len(profile_features)} customers")
profile_features.head()

84 customers


,customer_id,engagement_mean,interest_mean,platform,sentiment
0,A100,77.000000,4.400000,Instagram,Negative
1,A101,68.000000,1.000000,Twitter,Neutral
2,A102,51.000000,4.800000,LinkedIn,Neutral
3,A103,64.333333,2.866667,Instagram,Positive
4,A104,83.000000,2.933333,Twitter,Negative


### 5.2 Purchase-history features (from the transactions)

For each transaction, summarise what the same customer bought **before** it: number of earlier purchases, average amount and rating, and the share of each product category.

Only *earlier* purchases are used. That is what would be known when making a recommendation, and it keeps the current purchase out of its own features (otherwise the model could "see" the answer).

In [10]:
def add_history(tx):
    """Add features describing each customer's earlier purchases (strictly before the current one)."""
    tx = tx.sort_values(["purchase_date", "transaction_id"]).reset_index(drop=True)
    by_customer = tx.groupby("customer_id")

    n_prior = by_customer.cumcount()
    tx["prior_purchases"] = n_prior

    prior_spend = by_customer["purchase_amount"].cumsum() - tx["purchase_amount"]
    tx["prior_avg_amount"] = prior_spend / n_prior.replace(0, np.nan)

    rated = tx["customer_rating"].notna().astype(int)
    prior_rating_sum = tx["customer_rating"].fillna(0).groupby(tx["customer_id"]).cumsum() - tx["customer_rating"].fillna(0)
    prior_rating_n = rated.groupby(tx["customer_id"]).cumsum() - rated
    tx["prior_avg_rating"] = prior_rating_sum / prior_rating_n.replace(0, np.nan)

    for category in CATEGORIES:
        is_cat = (tx["product_category"] == category).astype(int)
        prior_cat = is_cat.groupby(tx["customer_id"]).cumsum() - is_cat
        tx[f"prior_share_{category}"] = (prior_cat / n_prior.replace(0, np.nan)).fillna(0.0)

    return tx


def build_model_table(tx):
    """Transactions + their history features, merged with the customer's profile features."""
    merged = add_history(tx).merge(profile_features, on="customer_id", how="left", indicator=True)
    return merged[merged["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)

### 5.3 Merge

Every transaction is matched to its customer's profile (one row per transaction). Transactions whose customer has no social profile cannot be used for a profile-based model and are left out.

In [11]:
with_history = add_history(transactions)
check = with_history.merge(profile_features, on="customer_id", how="left", indicator=True)
unmatched = check[check["_merge"] == "left_only"]
print(f"Transactions with a matching profile:  {(check['_merge'] == 'both').sum()}")
print(f"Transactions without a profile (left out): {len(unmatched)}  ({unmatched['customer_id'].nunique()} customers)")

model_df = build_model_table(transactions)
print(f"\nMerged table: {model_df.shape[0]} rows x {model_df.shape[1]} columns")
print(f"Duplicate rows: {model_df.duplicated().sum()} | 'Unknown' categories: {(model_df['product_category'] == 'Unknown').sum()}")
model_df.head()

Transactions with a matching profile:  117
Transactions without a profile (left out): 33  (14 customers)

Merged table: 117 rows x 18 columns
Duplicate rows: 0 | 'Unknown' categories: 0


,transaction_id,purchase_amount,purchase_date,product_category,customer_rating,customer_id,prior_purchases,prior_avg_amount,prior_avg_rating,prior_share_Books,prior_share_Clothing,prior_share_Electronics,prior_share_Groceries,prior_share_Sports,engagement_mean,interest_mean,platform,sentiment
0,1001,408,2024-01-01,Sports,2.3,A151,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,71.666667,2.166667,Twitter,Neutral
1,1002,332,2024-01-02,Electronics,4.2,A192,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,60.000000,4.300000,Instagram,Positive
2,1003,442,2024-01-03,Electronics,2.1,A114,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,87.000000,4.800000,Facebook,Negative
3,1005,64,2024-01-05,Clothing,1.3,A160,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,72.666667,2.500000,Instagram,Neutral
4,1006,395,2024-01-06,Groceries,1.1,A120,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,77.000000,2.666667,Instagram,Positive


In [12]:
print("Missing values in the merged table:")
print(model_df.isna().sum()[lambda s: s > 0])
print("\n(prior_avg_amount / prior_avg_rating are empty for a customer's first purchase; customer_rating for unrated purchases.)")
print("\nCategory counts:")
print(model_df["product_category"].value_counts())

Missing values in the merged table:
customer_rating      8
prior_avg_amount    61
prior_avg_rating    63
dtype: int64

(prior_avg_amount / prior_avg_rating are empty for a customer's first purchase; customer_rating for unrated purchases.)

Category counts:
product_category
Sports         28
Electronics    27
Clothing       22
Groceries      20
Books          20
Name: count, dtype: int64


## 6. Modelling

**Features**
- *Profile:* `engagement_mean`, `interest_mean`, `platform`, `sentiment`
- *History:* `prior_purchases`, `prior_avg_amount`, `prior_avg_rating`, `prior_share_<category>`

The current transaction's own amount, rating and date are **not** features: they are only known after the purchase.

**Evaluation.** With only ~117 rows, one train/test split is too noisy to trust. Models are scored with 5-fold cross-validation **repeated 10 times**, and each customer is kept entirely in one fold (so the model is never tested on a customer it has already seen). Every model is compared with a *dummy* model that just predicts category frequencies — a model is only useful if it beats that.

In [13]:
PROFILE_NUM = ["engagement_mean", "interest_mean"]
PROFILE_CAT = ["platform", "sentiment"]
HISTORY_NUM = ["prior_purchases", "prior_avg_amount", "prior_avg_rating"] + [f"prior_share_{c}" for c in CATEGORIES]

X_all = model_df[PROFILE_NUM + PROFILE_CAT + HISTORY_NUM]
y_all = model_df["product_category"]
groups = model_df["customer_id"]


def make_pipeline(estimator, numeric, categorical=()):
    """Impute + scale numeric inputs, one-hot encode categorical ones, then fit the estimator."""
    parts = [("num", Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())]), list(numeric))]
    if categorical:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical)))
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", estimator)])


def evaluate(estimator, numeric, categorical=(), data=None, n_repeats=10):
    """Repeated, customer-grouped cross-validation. Returns mean/std of accuracy, top-3 accuracy and log loss."""
    X, y, g = (X_all, y_all, groups) if data is None else data
    results = []
    for repeat in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + repeat)
        proba = np.zeros((len(X), len(CATEGORIES)))
        for train, test in cv.split(X, y, g):
            model = make_pipeline(estimator, numeric, categorical).fit(X.iloc[train], y.iloc[train])
            columns = np.searchsorted(CATEGORIES, model.classes_)   # keep columns aligned to CATEGORIES
            proba[np.ix_(test, columns)] = model.predict_proba(X.iloc[test])
        predicted = np.array(CATEGORIES)[proba.argmax(axis=1)]
        top3 = np.array(CATEGORIES)[np.argsort(-proba, axis=1)[:, :3]]
        results.append({
            "accuracy": accuracy_score(y, predicted),
            "top3": np.mean([truth in row for truth, row in zip(y, top3)]),
            "log_loss": log_loss(y, np.clip(proba, 1e-9, 1), labels=CATEGORIES),
        })
    r = pd.DataFrame(results)
    return {"Accuracy": r.accuracy.mean(), "Accuracy std": r.accuracy.std(),
            "Top-3 accuracy": r.top3.mean(), "Log loss": r.log_loss.mean()}

### 6.1 What does each data source add?

The same regularised logistic regression is trained on profile features only, history features only, and both.

In [14]:
lr = lambda C=0.03: LogisticRegression(C=C, max_iter=3000)

feature_sets = {
    "Profile only (social)":        (PROFILE_NUM, PROFILE_CAT),
    "History only (transactions)":  (HISTORY_NUM, ()),
    "Profile + history (merged)":   (PROFILE_NUM + HISTORY_NUM, PROFILE_CAT),
}

rows = {"Dummy (category frequencies)": evaluate(DummyClassifier(strategy="prior"), PROFILE_NUM, ())}
for name, (num, cat) in feature_sets.items():
    rows[f"Logistic regression - {name}"] = evaluate(lr(), num, cat)

feature_comparison = pd.DataFrame(rows).T.round(4)
feature_comparison

,Accuracy,Accuracy std,Top-3 accuracy,Log loss
Dummy (category frequencies),0.2179,0.0045,0.6530,1.6010
Logistic regression - Profile only (social),0.2803,0.0120,0.6350,1.6018
Logistic regression - History only (transactions),0.3239,0.0075,0.6863,1.5791
Logistic regression - Profile + history (merged),0.3359,0.0202,0.6906,1.5805


### 6.2 Model families and regularisation

On the merged features, compare model types, and a range of regularisation strengths `C` for logistic regression (smaller = simpler model, which helps on small data).

In [15]:
merged_num, merged_cat = PROFILE_NUM + HISTORY_NUM, PROFILE_CAT

candidates = {f"Logistic regression (C={C})": lr(C) for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]}
candidates["Random forest"] = RandomForestClassifier(
    n_estimators=300, max_depth=4, min_samples_leaf=5, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)
candidates["Gradient boosting"] = GradientBoostingClassifier(
    n_estimators=60, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=RANDOM_STATE)

model_comparison = pd.DataFrame(
    {name: evaluate(est, merged_num, merged_cat, n_repeats=5) for name, est in candidates.items()}
).T.round(4).sort_values("Log loss")
model_comparison

,Accuracy,Accuracy std,Top-3 accuracy,Log loss
Logistic regression (C=0.01),0.3470,0.0197,0.6598,1.5817
Logistic regression (C=0.03),0.3316,0.0164,0.6940,1.5846
Logistic regression (C=0.003),0.3043,0.0130,0.6427,1.5922
Random forest,0.2667,0.0164,0.6547,1.6314
Logistic regression (C=0.1),0.2855,0.0130,0.6786,1.6523
Logistic regression (C=0.3),0.2547,0.0140,0.6632,1.8140
Gradient boosting,0.1966,0.0249,0.6171,1.9013
Logistic regression (C=1),0.2410,0.0072,0.6513,2.1063


In [16]:
best_name = model_comparison.index[0]
best_estimator = candidates[best_name]
print("Selected:", best_name)

baseline = feature_comparison.loc["Dummy (category frequencies)"]
selected = pd.Series(evaluate(best_estimator, merged_num, merged_cat))
summary = pd.DataFrame({"Dummy baseline": baseline, "Selected model": selected}).T.round(4)
summary

Selected: Logistic regression (C=0.01)


,Accuracy,Accuracy std,Top-3 accuracy,Log loss
Dummy baseline,0.2179,0.0045,0.653,1.6010
Selected model,0.3444,0.0171,0.665,1.5801


### 6.3 Is the improvement real, or just luck?

With ~117 rows a gain over the baseline could be chance. To check, the product categories are **shuffled** across transactions (which destroys any real link between customer and category), the history features are rebuilt from the shuffled data, and the same model is scored. Repeating this many times gives the scores you would get from pure noise. If the real score sits well outside that range, the signal is real.

In [17]:
rng = np.random.default_rng(RANDOM_STATE)
null_scores = []
for _ in range(50):
    shuffled = transactions.copy()
    shuffled["product_category"] = rng.permutation(shuffled["product_category"].values)
    d = build_model_table(shuffled)
    null_scores.append(evaluate(
        best_estimator, merged_num, merged_cat,
        data=(d[PROFILE_NUM + PROFILE_CAT + HISTORY_NUM], d["product_category"], d["customer_id"]), n_repeats=1))
null_df = pd.DataFrame(null_scores)

real = selected
p_acc = (null_df["Accuracy"] >= real["Accuracy"]).mean()
p_ll = (null_df["Log loss"] <= real["Log loss"]).mean()

print(f"Accuracy  - real: {real['Accuracy']:.3f} | shuffled labels: {null_df['Accuracy'].mean():.3f} +/- {null_df['Accuracy'].std():.3f} | p = {p_acc:.3f}")
print(f"Log loss  - real: {real['Log loss']:.4f} | shuffled labels: {null_df['Log loss'].mean():.4f} +/- {null_df['Log loss'].std():.4f} | p = {p_ll:.3f}")

Accuracy  - real: 0.344 | shuffled labels: 0.239 +/- 0.031 | p = 0.000
Log loss  - real: 1.5801 | shuffled labels: 1.6043 +/- 0.0156 | p = 0.080


### 6.4 Reading the results

Look at the three tables together:
- **Which source helps:** compare the "Profile only", "History only" and "Profile + history" rows in 6.1.
- **Against the baseline:** the selected model should beat the dummy row on accuracy and log loss.
- **Against chance:** the p-values in 6.3 say how often shuffled data does as well. A small p-value (e.g. below 0.05) means the improvement is unlikely to be luck.

The accuracy is modest because product choice depends on much more than these few columns, and the dataset is small. The scores are estimates from ~117 rows, so treat them as indicative rather than exact.

### 6.5 Which features matter

In [18]:
final_model = make_pipeline(best_estimator, merged_num, merged_cat).fit(X_all, y_all)
print(f"Final model: {best_name}, fitted on all {len(X_all)} rows")

if hasattr(final_model.named_steps["model"], "coef_"):
    names = final_model.named_steps["prep"].get_feature_names_out()
    influence = np.abs(final_model.named_steps["model"].coef_).mean(axis=0)
    feature_influence = (
        pd.DataFrame({"Feature": [n.split("__", 1)[1] for n in names], "Influence (mean |coefficient|)": influence})
        .sort_values("Influence (mean |coefficient|)", ascending=False)
        .reset_index(drop=True).round(4)
    )
else:
    importances = final_model.named_steps["model"].feature_importances_
    names = final_model.named_steps["prep"].get_feature_names_out()
    feature_influence = (
        pd.DataFrame({"Feature": [n.split("__", 1)[1] for n in names], "Importance": importances})
        .sort_values("Importance", ascending=False).reset_index(drop=True).round(4)
    )
feature_influence.head(10)

Final model: Logistic regression (C=0.01), fitted on all 117 rows


,Feature,Influence (mean |coefficient|)
0,prior_purchases,0.0465
1,interest_mean,0.0445
2,prior_share_Clothing,0.0419
3,prior_avg_rating,0.0401
4,prior_share_Groceries,0.0369
5,prior_share_Electronics,0.0327
6,prior_share_Sports,0.0264
7,prior_share_Books,0.0252
8,prior_avg_amount,0.0228
9,engagement_mean,0.0212


## 7. Recommendation engine

Takes a customer's social profile and, optionally, their earlier purchases, and returns the most likely product category plus the top 3 with probabilities. For a new customer with no purchases, leave `past_purchases` empty.

`past_purchases` is a list of dicts with the keys `product_category`, `purchase_amount` and (optionally) `customer_rating`.

In [19]:
def recommend_product(engagement, interest, platform, sentiment, past_purchases=None, model=final_model, top_k=3):
    """Return the recommended product category, its confidence and the top-k alternatives."""
    past = pd.DataFrame(past_purchases or [], columns=["product_category", "purchase_amount", "customer_rating"])
    n_prior = len(past)

    row = {
        "engagement_mean": float(engagement),
        "interest_mean": float(interest),
        "platform": platform,
        "sentiment": sentiment,
        "prior_purchases": n_prior,
        "prior_avg_amount": past["purchase_amount"].mean() if n_prior else np.nan,
        "prior_avg_rating": past["customer_rating"].mean() if past["customer_rating"].notna().any() else np.nan,
    }
    for category in CATEGORIES:
        row[f"prior_share_{category}"] = float((past["product_category"] == category).mean()) if n_prior else 0.0

    probabilities = model.predict_proba(pd.DataFrame([row])[X_all.columns])[0]
    order = np.argsort(probabilities)[::-1]
    classes = model.classes_
    return {
        "recommended_category": classes[order[0]],
        "confidence_score": round(float(probabilities[order[0]]), 2),
        "top_categories": {classes[i]: round(float(probabilities[i]), 2) for i in order[:top_k]},
    }

In [20]:
# New customer (no purchase history)
print(recommend_product(engagement=74, interest=4.9, platform="LinkedIn", sentiment="Positive"))

# Returning customer with two earlier purchases
print(recommend_product(
    engagement=90, interest=4.8, platform="Twitter", sentiment="Positive",
    past_purchases=[
        {"product_category": "Electronics", "purchase_amount": 320, "customer_rating": 4.5},
        {"product_category": "Sports", "purchase_amount": 150, "customer_rating": 3.0},
    ],
))

{'recommended_category': 'Electronics', 'confidence_score': 0.31, 'top_categories': {'Electronics': 0.31, 'Sports': 0.21, 'Clothing': 0.18}}
{'recommended_category': 'Sports', 'confidence_score': 0.24, 'top_categories': {'Sports': 0.24, 'Electronics': 0.23, 'Clothing': 0.21}}


## 8. Save the cleaned data and the model

**Cleaned data** — the merged table used for modelling: one row per transaction, matched to its customer's profile, with no duplicate rows and no `Unknown` categories. Missing values remain only where there is genuinely nothing to compute (a first purchase has no earlier purchases; some purchases were not rated).

**Model** — the whole pipeline (imputing, scaling, encoding and the classifier) saved as one file with `joblib`, with the feature names and scikit-learn version it was built with.

In [21]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_PATH = PROCESSED_DIR / "cleaned_customer_data.csv"

id_cols = ["customer_id", "transaction_id", "purchase_date", "purchase_amount", "customer_rating", "product_category"]
cleaned_export = model_df[id_cols + PROFILE_NUM + PROFILE_CAT + HISTORY_NUM].copy()
cleaned_export["purchase_date"] = cleaned_export["purchase_date"].dt.date
cleaned_export = cleaned_export.round(4)
cleaned_export.to_csv(CLEANED_PATH, index=False)

check = pd.read_csv(CLEANED_PATH)
print(f"Saved {len(check)} rows x {check.shape[1]} columns to {CLEANED_PATH}")
print("Duplicate rows:", int(check.duplicated().sum()), "| 'Unknown' values:", int((check == "Unknown").sum().sum()))
check.head()

Saved 117 rows x 18 columns to data/processed/cleaned_customer_data.csv
Duplicate rows: 0 | 'Unknown' values: 0


,customer_id,transaction_id,purchase_date,purchase_amount,customer_rating,product_category,engagement_mean,interest_mean,platform,sentiment,prior_purchases,prior_avg_amount,prior_avg_rating,prior_share_Books,prior_share_Clothing,prior_share_Electronics,prior_share_Groceries,prior_share_Sports
0,A151,1001,2024-01-01,408,2.3,Sports,71.6667,2.1667,Twitter,Neutral,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0
1,A192,1002,2024-01-02,332,4.2,Electronics,60.0000,4.3000,Instagram,Positive,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0
2,A114,1003,2024-01-03,442,2.1,Electronics,87.0000,4.8000,Facebook,Negative,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0
3,A160,1005,2024-01-05,64,1.3,Clothing,72.6667,2.5000,Instagram,Neutral,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0
4,A120,1006,2024-01-06,395,1.1,Groceries,77.0000,2.6667,Instagram,Positive,0,NaN,NaN,0.0,0.0,0.0,0.0,0.0


In [22]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "product_recommender.joblib"

bundle = {
    "pipeline": final_model,
    "model_name": best_name,
    "classes": list(final_model.classes_),
    "feature_columns": list(X_all.columns),
    "categories": CATEGORIES,
    "sklearn_version": sklearn.__version__,
    "cv_metrics": selected.round(4).to_dict(),
}
joblib.dump(bundle, MODEL_PATH)
print(f"Saved model to {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.1f} KB)")

# Reload and confirm identical predictions
loaded = joblib.load(MODEL_PATH)
same = np.allclose(loaded["pipeline"].predict_proba(X_all), final_model.predict_proba(X_all))
print(f"Loaded '{loaded['model_name']}' (scikit-learn {loaded['sklearn_version']}) | matches in-memory model: {same}")
print(recommend_product(engagement=74, interest=4.9, platform="LinkedIn", sentiment="Positive", model=loaded["pipeline"]))

Saved model to models/product_recommender.joblib (5.9 KB)
Loaded 'Logistic regression (C=0.01)' (scikit-learn 1.9.1) | matches in-memory model: True
{'recommended_category': 'Electronics', 'confidence_score': 0.31, 'top_categories': {'Electronics': 0.31, 'Sports': 0.21, 'Clothing': 0.18}}
